In [7]:
import random
import os

In [2]:
CONFIG_FOLDER = "/home/vlad/Documents/SublimeMerge/dragonfly-scripts/simulate_dfly_queue_sizes/configs/"

def compute_Z(h, buckets):
    cnt_nodes, cnt_groups, group_size = (h**2 + 1) * h**2, h**2 + 1, h**2
    Z = [[0 for j in range(cnt_groups)] for i in range(cnt_groups)]
    for row in buckets:
        for buck in row:
            for i in range(h):
                for j in range(i+1, h):
                    Z[min(buck[i], buck[j])][max(buck[i], buck[j])] += 1
    return Z

def get_score(h, buckets):
    cnt_nodes, cnt_groups, group_size = (h**2 + 1) * h**2, h**2 + 1, h**2
    Z = compute_Z(h, buckets)
    scor = 0
    for i in range(cnt_groups):
        for j in range(i+1, cnt_groups):
            scor -= abs(Z[i][j] - (h - 1))
    return scor

def build_default_config(h):
    cnt_nodes, cnt_groups, group_size = (h**2 + 1) * h**2, h**2 + 1, h**2
    buckets = [[[] for j in range(h)] for i in range(cnt_groups)]
    for src_group in range(cnt_groups):
        for dst_group in range(cnt_groups):
            if src_group != dst_group:
                src_sw = (dst_group - (1 if src_group < dst_group else 0)) // h
                dst_sw = (src_group - (0 if src_group < dst_group else 1)) // h
                buckets[src_group][src_sw].append(dst_group)
    return buckets

def read_buckets(h, fname):
    make_buckets = lambda v: [[v[j] for j in range(i*h, (i+1)*h)] for i in range(h)]
    with open(fname) as fin:
        return [make_buckets(list(map(int, line.strip().split()))) for line in fin.readlines()]
    
def get_random_buckets(h, nt):
    cnt_groups = h**2 + 1
    
    split_in_buckets = lambda v: [[v[j] for j in range(i, i+h)] for i in range(0, cnt_groups-1, h)]
    
    return [
        [
            split_in_buckets(random.sample(list(range(i)) + list(range(i+1, cnt_groups)), cnt_groups - 1))
            for i in range(cnt_groups)
        ] for _ in range(nt)
    ]

In [5]:
get_random_buckets(2, 1)[0]

[[[2, 3], [4, 1]],
 [[2, 4], [0, 3]],
 [[3, 4], [1, 0]],
 [[4, 1], [0, 2]],
 [[0, 2], [1, 3]]]

In [12]:
nt = 10**2
hs = [6, 8, 10]
cnt_want_same = 5

for h in hs:
    arr_buckets = get_random_buckets(h, nt)
    scores_ind = [(get_score(h, buckets), i) for i, buckets in enumerate(arr_buckets)]
    
    mean_score = sum(map(lambda p: p[0], scores_ind)) / nt
    scores_ind.sort(key = lambda p: abs(p[0] - mean_score))
    
    i = 0
    while i < nt:
        j = i
        while j < nt and scores_ind[j][0] == scores_ind[i][0]:
            j += 1
        if j - i + 1 >= cnt_want_same:
            print(f"{h = }, {j-i+1 = }, {mean_score = }, {scores_ind[i][0] = }")
            
            for z in range(i, i+cnt_want_same):
                buckets = arr_buckets[scores_ind[z][1]]
                with open(os.path.join(CONFIG_FOLDER, f"k_{2*h}/config_{z-i+1}_score_{scores_ind[i][0]}.txt"), 'w') as fout:
                    for row in buckets:
                        fout.write(' '.join([str(x) for spine in row for x in spine]) + '\n')
            break
        i = j

h = 6, j-i+1 = 6, mean_score = -1078.14, scores_ind[i][0] = -1080
h = 8, j-i+1 = 6, mean_score = -4082.42, scores_ind[i][0] = -4082
h = 10, j-i+1 = 5, mean_score = -11413.9, scores_ind[i][0] = -11282
